## Content-Based Recommender C (with metadata and review sentiment analysis)

### Loading Data

In [1]:
import pandas as pd
import numpy as np

#resturants
df_rests = pd.read_csv(
    "restaurants_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_rests.columns = df_rests.iloc[0]
df_rests = df_rests[1:]
df_rests.reset_index(drop=True)

#test
df_test = pd.read_csv(
    "test_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_test.columns = df_test.iloc[0]
df_test = df_test[1:]
df_test.reset_index(drop=True)

#train
df_train = pd.read_csv(
    "train_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_train.columns = df_train.iloc[0]
df_train = df_train[1:]
df_train.reset_index(drop=True)

,review_id,user_id,business_id,stars,datetime,text
0,ixa--i8zAivs8rD10tleZQ,-0EcgtUXe1rzrkmdih_tYg,VgAKmXE8B7J0I_O_R13UKQ,4.0,2018-07-03 03:24:32,I lived here before and not once that I came t...
1,MHj1HEPM5Bv7_UhoiwOSGA,-0EcgtUXe1rzrkmdih_tYg,YrNtBUOUOYwmRZ_UVwH8iQ,4.0,2018-07-04 15:47:17,Totally a great experienced. Been coming to S...
2,_sTp9AsEu60ORSqqb9juJg,-0EcgtUXe1rzrkmdih_tYg,Aes-0Q_guDeYewMapFs_vg,4.0,2018-07-06 05:38:42,"Great outdoor seating, the servers are attenti..."
3,2Tcu86xzIADtfuy5jeio9A,-0EcgtUXe1rzrkmdih_tYg,9ugpNKKhnYRa51qXoxUw_A,3.0,2018-07-06 05:49:49,Went for the first time. The lady in the fron...
4,J6X5I_LQnii7QeIW79gBVg,-0EcgtUXe1rzrkmdih_tYg,DOfiulOub9hVPBCtiDl9Fw,3.0,2018-07-06 06:05:15,"Pizza was great, servers are friendly and atte..."
...,...,...,...,...,...,...
41576,hJ6_1V1gLMuWMUfPeaGomA,zyt0joW7uNeQof5tthQAHg,JjmmSW_QQh2Db4fuIEMATA,4.0,2013-09-08 02:32:56,I never had such a good pizza so fast! We had ...
41577,wqUCvQf_bP_gQxrFRz94LA,zyt0joW7uNeQof5tthQAHg,Aes-0Q_guDeYewMapFs_vg,3.0,2013-09-09 03:56:41,We had lunch at the longboard grill. The food ...
41578,CufxkovWUbBqcj8avCai_g,zyt0joW7uNeQof5tthQAHg,0CHIbqSkGWBr2KMkIUocEA,4.0,2013-09-09 04:31:33,So first let me say that this restaurant is in...
41579,0yQfK9Ci7fbX850-LEyJbw,zyt0joW7uNeQof5tthQAHg,U2hkeMI-q4cS35QolJYN0A,2.0,2013-09-10 02:46:32,"So if this is, as other reviewers state, the b..."


### Embedding restaurant categories

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


#FIRST I WANT TO EMBED THE ITEMS (THE RESTAURANTS) AND GET VECTORS FOR EACH
df_rests = df_rests.reset_index(drop=True)
embedder_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder_model.encode(df_rests["categories"])
df_rests["category_embeddings"] = list(embeddings)

/Users/troyhealey/PycharmProjects/RecomSystemsClass/.venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
2026-04-12 10:47:06.565388: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/troyhealey/PycharmProjects/RecomSystemsClass/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more detai

### Adding attribute features

In [3]:
#on top of the categorical embeddings, I want to add to each embedding a few values connoting attribute values

#first clean values for the attribute column
import ast
def str_to_dict(s):
    try:
        return ast.literal_eval(s)
    except: #need this or else it won't run
        return {}
df_rests['attributes_dict'] = df_rests['attributes'].apply(str_to_dict)

def clean_values(d):
    return {k: (v.strip("'\"") if isinstance(v, str) else v) for k, v in d.items()}
df_rests['attributes_dict'] = df_rests['attributes_dict'].apply(clean_values)

#then use RestaurantsPriceLevel2, GoodForKids, and RestaurantsAttire; these are all good indicators because people feel strongly about prices, about their families, and about "comfortability" of a restaurant. A person who wants to eat cheap will really appreciate a cheap recommendation, a person who has kids will really appreciate a restaurant that caters to them, and a person who likes to be comfortable/homey will really appreciate a restaurant recommendation that feels like home.
df_rests['good_for_kids'] = 0
maskgfk = df_rests['attributes_dict'].apply(lambda d: d.get('GoodForKids') == 'True')
df_rests.loc[maskgfk, 'good_for_kids'] = 1

df_rests['rest_p_level_1'] = 0
df_rests['rest_p_level_2'] = 0
mask1 = df_rests['attributes_dict'].apply(lambda d: d.get('RestaurantsPriceRange2') == '1')
mask2 = df_rests['attributes_dict'].apply(lambda d: d.get('RestaurantsPriceRange2') == '2')

df_rests.loc[mask1, 'rest_p_level_1'] = 1
df_rests.loc[mask2, 'rest_p_level_2'] = 1

df_rests['casual'] = 0
maskcas = df_rests['attributes_dict'].apply(lambda d: d.get('RestaurantsAttire') == 'casual')
df_rests.loc[maskcas, 'casual'] = 1

#then, you want these values to enter the category embedding to make one big embedding, so you can calculate similarities of single vectors.
import numpy as np

df_rests["new_embedding"] = df_rests.apply(lambda row: np.concatenate([row["category_embeddings"],
        [row["good_for_kids"], row["rest_p_level_1"], row["rest_p_level_2"], row['casual']]]), axis=1)

### Aggregate reviews and embed

In [4]:
# group df_train by business_id

# Aggregate all review texts per restaurant from training data
review_text_by_restaurant = df_train.groupby("business_id")["text"].apply(
    lambda texts: " ".join(texts.dropna())
).reset_index()
review_text_by_restaurant.columns = ["business_id", "aggregated_text"]

# Embed the aggregated review texts
review_embeddings = embedder_model.encode(review_text_by_restaurant["aggregated_text"].tolist())
review_text_by_restaurant["review_embedding"] = list(review_embeddings)

# Merge into restaurant dataframe
df_rests_b = df_rests.merge(
    review_text_by_restaurant[["business_id", "review_embedding"]],
    on="business_id",
    how="left"
)

# Fill restaurants with no reviews with a zero vector (same size as the model output)
embedding_size = review_embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
df_rests_b["review_embedding"] = df_rests_b["review_embedding"].apply(
    lambda x: x if isinstance(x, np.ndarray) else np.zeros(embedding_size)
)

### Extract sentiment from reviews

In [5]:
from transformers import pipeline

sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def get_sentiment_score(text):
    """
    Returns a sentiment score between -1 (very negative) and 1 (very positive).
    """
    result = sentiment_analyzer(text[:512])[0]
    score = result['score']
    if result['label'] == 'NEGATIVE':
        score = -score
    return score

review_text_by_restaurant["sentiment_score"] = review_text_by_restaurant["aggregated_text"].apply(get_sentiment_score)

/Users/troyhealey/PycharmProjects/RecomSystemsClass/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Add sentiment score to embedding

In [6]:
df_rests_c = df_rests_b.merge(
    review_text_by_restaurant[["business_id", "sentiment_score"]],
    on="business_id",
    how="left"
)
df_rests_c["sentiment_score"] = df_rests_c["sentiment_score"].fillna(0.0)

In [7]:
df_rests_c["new_embedding"] = df_rests_c.apply(
    lambda row: np.concatenate([
        row["category_embeddings"],
        [row["good_for_kids"], row["rest_p_level_1"], row["rest_p_level_2"], row["casual"]],
        row["review_embedding"],
        [row["sentiment_score"]]        # add sentiment score
    ]), axis=1
)

### Build user profiles

In [8]:
df_train['stars'] = pd.to_numeric(df_train['stars'])
user_means = df_train.groupby("user_id")["stars"].mean()
df_train = df_train.merge(user_means.rename("user_mean"), on="user_id")
df_train["adjusted_rating"] = df_train["stars"] - df_train["user_mean"]

merge = df_train.merge(df_rests_c, on="business_id")
merge = merge.set_index("business_id")

#now we want to get the user profiles
user_profiles = {}
for user, group in df_train.groupby("user_id"):
    vectors = merge.loc[group["business_id"], "new_embedding"]
    user_profiles[user] = np.vstack(vectors).mean(axis=0)

### Recommendation function

In [9]:
#get recommendations
from sklearn.metrics.pairwise import cosine_similarity

def recommend_restaurants(user_id, user_embeddings, restaurants_df, k):
    user_vec = user_embeddings[user_id].reshape(1, -1)
    restaurant_matrix = np.vstack(restaurants_df['new_embedding'].values)

    similarities = cosine_similarity(user_vec, restaurant_matrix)[0]
    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = restaurants_df[['business_id', 'name']].iloc[top_k_idx].copy()
    results['similarity'] = similarities[top_k_idx]

    return results

### Hit and NDCG formulas

In [10]:
#now we can do accuracy metrics. first is NDCG@k
def ndcg_at_k(recommended_ids, true_id, k):
    try:
        rank = recommended_ids.index(true_id) + 1
    except ValueError:
        return 0.0

    return 1 / np.log2(rank + 1)

def evaluate_ndcg(user_embeddings, restaurants_df, test_df, k):
    scores = []

    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']

        recs = recommend_restaurants(user_id,user_embeddings,restaurants_df,k)
        recommended_ids = recs['business_id'].tolist()
        score = ndcg_at_k(recommended_ids, true_business_id, k)
        scores.append(score)

    return np.mean(scores)

#then hit@k
def hit_at_k(recommended_ids, true_id):
    return int(true_id in recommended_ids)

def evaluate_hit_at_k(user_embeddings, restaurants_df, test_df, k):
    hits = []
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']
        recs = recommend_restaurants(user_id, user_embeddings, restaurants_df, k)

        recommended_ids = recs['business_id'].tolist()

        hit = hit_at_k(recommended_ids, true_business_id)
        hits.append(hit)

    return np.mean(hits)

### Printing Evaluation Metrics

In [11]:
print("Content-Based Recommender C (Metadata and Review Sentiment Analysis): Evaluation Metrics")
for k in [10, 20, 30]:
    print(f"Hit@{k}:", evaluate_hit_at_k(user_profiles, df_rests_c, df_test, k), f"\t\tNDCG@{k}:", evaluate_ndcg(user_profiles, df_rests_c, df_test, k))

Content-Based Recommender C (Metadata and Review Sentiment Analysis): Evaluation Metrics
Hit@10: 0.02353676317433868 		NDCG@10: 0.009352737019443495
Hit@20: 0.04790668610706103 		NDCG@20: 0.015442050702578926
Hit@30: 0.07269318891897521 		NDCG@30: 0.02070598901139927
